# DeepGym Quickstart

**DeepGym** — RL training environments with verifiable rewards for coding agents.

Give it model-generated code. It runs the code, verifies it, returns a score.
That score plugs into TRL, verl, OpenRLHF, or any RL training loop.

[![GitHub](https://img.shields.io/badge/GitHub-DeepGym-black)](https://github.com/DeepGym/deepgym)
[![PyPI](https://img.shields.io/pypi/v/deepgym)](https://pypi.org/project/deepgym/)

This notebook covers:
1. Score a single solution
2. Batch scoring (GRPO-style)
3. Per-test-case reward traces
4. Drop into TRL GRPOTrainer
5. Write your own verifier
6. Browse built-in environments

In [ ]:
# Install DeepGym
!pip install deepgym -q

## 1. Score a single solution

Load a built-in environment, submit a solution, get a score.

In [ ]:
from deepgym import DeepGym, load_environment

dg = DeepGym(mode='local')          # no Daytona needed for local dev
env = load_environment('coin_change')

correct_solution = '''
def coin_change(coins, amount):
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    for coin in coins:
        for x in range(coin, amount + 1):
            dp[x] = min(dp[x], dp[x - coin] + 1)
    return dp[amount] if dp[amount] != float('inf') else -1
'''

result = dg.run(env, model_output=correct_solution)
print(f'Score:  {result.score}')
print(f'Passed: {result.passed}')
print(f'Time:   {result.execution_time_ms:.0f}ms')

In [ ]:
# A wrong solution scores 0.0
wrong_solution = '''
def coin_change(coins, amount):
    return -1  # always wrong
'''

result = dg.run(env, model_output=wrong_solution)
print(f'Score:  {result.score}')   # 0.0
print(f'Passed: {result.passed}')  # False

## 2. Batch scoring for GRPO

GRPO needs to score N completions for the same prompt and compute advantages.
DeepGym runs them in parallel.

In [ ]:
# Simulate 8 model completions (mix of correct and wrong)
solutions = [
    # Correct DP solution
    '''
def coin_change(coins, amount):
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    for coin in coins:
        for x in range(coin, amount + 1):
            dp[x] = min(dp[x], dp[x - coin] + 1)
    return dp[amount] if dp[amount] != float('inf') else -1
''',
    # Greedy (wrong for some inputs)
    '''
def coin_change(coins, amount):
    coins.sort(reverse=True)
    count = 0
    for coin in coins:
        count += amount // coin
        amount %= coin
    return count if amount == 0 else -1
''',
    # Always returns -1
    'def coin_change(coins, amount): return -1',
    # Correct BFS solution
    '''
from collections import deque
def coin_change(coins, amount):
    if amount == 0: return 0
    visited = {0}
    q = deque([(0, 0)])
    while q:
        curr, steps = q.popleft()
        for coin in coins:
            nxt = curr + coin
            if nxt == amount: return steps + 1
            if nxt < amount and nxt not in visited:
                visited.add(nxt)
                q.append((nxt, steps + 1))
    return -1
''',
] * 2  # 8 total

batch = dg.run_batch(env, solutions, max_parallel=8)
scores = [r.score for r in batch.results]

print(f'Scores: {[f"{s:.2f}" for s in scores]}')
print(f'Passed: {batch.passed}/{batch.total}')
print(f'Avg:    {batch.avg_score:.3f}')

# GRPO advantage computation
mean_score = sum(scores) / len(scores)
std_score = (sum((s - mean_score) ** 2 for s in scores) / len(scores)) ** 0.5
advantages = [(s - mean_score) / (std_score + 1e-8) for s in scores]
print(f'Advantages: {[f"{a:.2f}" for a in advantages]}')

## 3. Per-test-case reward traces

DeepGym returns **which specific tests passed and failed** — not just an overall score.
This gives GRPO a denser training signal.

In [ ]:
result = dg.run(env, model_output=correct_solution)

if result.cases:
    print(f'Total test cases: {len(result.cases)}')
    print()
    for case in result.cases[:5]:  # show first 5
        status = 'PASS' if case.passed else 'FAIL'
        print(f'  [{status}] {case.id}: {case.input_summary}')
        if not case.passed and case.error:
            print(f'         Error: {case.error}')
else:
    print('No per-test traces (verifier does not emit cases)')

In [ ]:
from deepgym import RewardFunction

reward_fn = RewardFunction(env=env, max_parallel=8)

# Per-test-case breakdown for each completion
per_test = reward_fn.per_test_rewards(solutions[:4])
for i, breakdown in enumerate(per_test):
    print(f'Solution {i}: overall={breakdown["overall"]:.2f}, '
          f'tests={sum(1 for k,v in breakdown.items() if k != "overall" and v == 1.0)}/'
          f'{sum(1 for k in breakdown if k != "overall")} passed')

## 4. Drop into TRL GRPOTrainer

One-liner integration with Hugging Face TRL.

In [ ]:
# Install TRL if not present
# !pip install trl -q

from deepgym.integrations.trl import make_trl_reward_fn

env = load_environment('two_sum')
reward_fn = make_trl_reward_fn(env)  # returns Callable[[list[str]], list[float]]

# Test it directly
test_completions = [
    'def two_sum(nums, target):\n    seen = {}\n    for i, n in enumerate(nums):\n        if target - n in seen:\n            return [seen[target - n], i]\n        seen[n] = i',
    'def two_sum(nums, target): return []',  # wrong
]
scores = reward_fn(test_completions)
print(f'TRL reward scores: {scores}')

# In a real training loop:
# from trl import GRPOTrainer
# trainer = GRPOTrainer(
#     model='Qwen/Qwen2-0.5B-Instruct',
#     reward_funcs=[reward_fn],
#     train_dataset=dataset,
# )
# trainer.train()

## 5. Write your own verifier

The verifier is just Python code that returns a float 0.0-1.0 (or bool or dict).
DeepGym wraps it into the JSON protocol automatically.

In [ ]:
from deepgym import DeepGym, Environment

# Verifier body: receives solution_path and optional test_cases_path
# Return float, bool, or dict with {score, passed, details, cases}
verifier_code = '''
import importlib.util, sys
spec = importlib.util.spec_from_file_location("sol", solution_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

cases = [
    (2, 3, 5),
    (0, 0, 0),
    (-1, 1, 0),
    (100, 200, 300),
]

passed = 0
case_results = []
for a, b, expected in cases:
    try:
        got = mod.add(a, b)
        ok = got == expected
    except Exception as e:
        ok = False
    if ok:
        passed += 1
    case_results.append({
        "id": f"test_{a}+{b}",
        "passed": ok,
        "score": 1.0 if ok else 0.0,
        "input_summary": f"add({a}, {b})",
        "expected_summary": str(expected),
    })

score = passed / len(cases)
return {"score": score, "passed": score == 1.0, "details": f"{passed}/{len(cases)} passed", "cases": case_results}
'''

env = Environment(
    task='Write a function `add(a, b)` that returns the sum of two numbers.',
    verifier_code=verifier_code,
)

dg = DeepGym(mode='local')
result = dg.run(env, model_output='def add(a, b):\n    return a + b')
print(f'Score:  {result.score}')
print(f'Passed: {result.passed}')
for c in (result.cases or []):
    print(f'  {c.id}: {"PASS" if c.passed else "FAIL"}')

## 6. Browse built-in environments

In [ ]:
from deepgym import list_environments

envs = list_environments()
print(f'{len(envs)} built-in environments:')
for name in sorted(envs):
    print(f'  {name}')

In [ ]:
# After importing benchmarks (run these scripts first):
# !python scripts/import_humaneval.py      # 164 HumanEval problems
# !python scripts/import_mbpp.py           # 500 MBPP problems
# !python scripts/import_bigcodebench.py   # 1,140 BigCodeBench problems

# Then:
# env = load_environment('HumanEval/0')   # first HumanEval problem
# env = load_environment('MBPP/1')        # first MBPP problem
print('Run the import scripts above to get 2,350+ benchmark environments.')

## 7. Gymnasium-style API

If you prefer a reset/step loop.

In [ ]:
from deepgym import DeepGymEnv, load_environment

env = load_environment('reverse_string')
gym_env = DeepGymEnv(env=env, mode='local')

obs = gym_env.reset()
print('Task:', obs.task[:80], '...')

solution = '''
def reverse_string(s):
    return s[::-1]
'''

obs, reward, terminated, info = gym_env.step(solution)
print(f'Reward: {reward:.2f}')
print(f'Done:   {terminated}')